In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import os
import re
from pathlib import Path

# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# =============================================================================
# 1. CARGA Y CONCATENACIÓN DE DATOS
# =============================================================================

def extraer_anio_y_ciclo(filename):
    """Extrae año y ciclo desde nombres como rendimiento_año_2025_ciclo_1_anon.xlsx."""
    texto = filename.lower()
    match_anio = re.search(r'(20\d{2})', texto)
    match_ciclo = re.search(r'ciclo[_\s-]*(\d)', texto)

    if not match_anio or not match_ciclo:
        raise ValueError(f"No se pudo parsear año/ciclo en: {filename}")

    return int(match_anio.group(1)), int(match_ciclo.group(1))


def cargar_y_concatenar(PATH, files):
    """Carga y concatena todos los archivos xls/xlsx del directorio."""
    datos_concatenados = pd.DataFrame()

    for i, filename in enumerate(sorted(files), 1):
        print(f"📄 Procesando archivo {i}/{len(files)}: {filename}")

        try:
            ruta_archivo = os.path.join(PATH, filename)
            lectura = pd.read_excel(ruta_archivo, sheet_name=0)
            lectura = lectura.copy()

            anio, ciclo = extraer_anio_y_ciclo(filename)
            lectura['Anho'] = anio
            lectura['Semestre'] = ciclo
            lectura['Archivo_Origen'] = filename

            datos_concatenados = pd.concat([datos_concatenados, lectura], axis=0, ignore_index=True)
            print(f"  ✅ Registros agregados: {len(lectura):,} | Total acumulado: {len(datos_concatenados):,}")

        except Exception as e:
            print(f"  ❌ Error al procesar {filename}: {e}")

    print(f"\n✅ Total de registros concatenados: {len(datos_concatenados):,}")
    return datos_concatenados

# =============================================================================
# 2. ANÁLISIS DESCRIPTIVO GENERAL
# =============================================================================

def analisis_descriptivo_general(df):
    """Realiza un análisis descriptivo completo de los datos."""
    print("="*80)
    print("📊 ANÁLISIS DESCRIPTIVO GENERAL DE LOS DATOS")
    print("="*80)

    print("\n" + "="*80)
    print("1. INFORMACIÓN BÁSICA DEL DATASET")
    print("="*80)

    print(f"📌 Total de registros: {len(df):,}")
    print(f"📌 Total de columnas: {len(df.columns)}")
    print(f"📌 Total de estudiantes únicos: {df['ALUMNO_ID'].nunique() if 'ALUMNO_ID' in df.columns else 'N/A'}")
    print(f"📌 Total de asignaturas únicas: {df['Asignatura'].nunique() if 'Asignatura' in df.columns else 'N/A'}")
    print(f"📌 Rango de años: {df['Anho'].min()} - {df['Anho'].max()}")
    print(f"📌 Semestres disponibles: {sorted(df['Semestre'].unique())}")

    print("\n" + "="*80)
    print("2. ESTRUCTURA DE LOS DATOS")
    print("="*80)

    print("\n📋 Primeras 5 filas:")
    display(df.head())

    print("\n📋 Últimas 5 filas:")
    display(df.tail())

    print("\n📋 Tipos de datos por columna:")
    print(df.dtypes)

    print("\n" + "="*80)
    print("3. ESTADÍSTICAS DESCRIPTIVAS")
    print("="*80)

    columnas_numericas = df.select_dtypes(include=[np.number]).columns
    print("\n📊 Estadísticas de columnas numéricas:")
    display(df[columnas_numericas].describe())

    columnas_categoricas = df.select_dtypes(include=['object']).columns
    print("\n📊 Columnas categóricas:")
    for col in columnas_categoricas[:10]:
        print(f"\n{col}:")
        print(f"  - Valores únicos: {df[col].nunique()}")
        print(f"  - Más frecuente: {df[col].mode().iloc[0] if not df[col].mode().empty else 'N/A'}")
        print(f"  - Frecuencia: {df[col].value_counts().iloc[0] if not df[col].empty else 0}")

    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing_Count': missing_data,
        'Missing_Percent': missing_percent
    }).loc[lambda x: x['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

    print("\n" + "="*80)
    print("4. ANÁLISIS DE DATOS FALTANTES")
    print("="*80)

    if len(missing_df) > 0:
        print(f"\n🔍 Columnas con datos faltantes ({len(missing_df)} columnas):")
        display(missing_df)
    else:
        print("✅ No hay datos faltantes en el dataset")

    duplicados = df.duplicated().sum()
    print("\n" + "="*80)
    print("5. ANÁLISIS DE REGISTROS DUPLICADOS")
    print("="*80)
    print(f"📌 Registros duplicados: {duplicados:,} ({duplicados/len(df)*100:.2f}%)")

    if duplicados > 0:
        print("\n🔍 Ejemplo de registros duplicados:")
        display(df[df.duplicated(keep=False)].head(5))

    year_semester_counts = df.groupby(['Anho', 'Semestre']).size().reset_index(name='Registros')
    print("\n" + "="*80)
    print("6. DISTRIBUCIÓN POR AÑO Y SEMESTRE")
    print("="*80)
    display(year_semester_counts)

    if 'Asignatura' in df.columns:
        print("\n📚 Top 10 asignaturas con más registros:")
        display(df['Asignatura'].value_counts().head(10))

    if 'Cod.Curso' in df.columns:
        print("\n📚 Número de asignaturas por nivel:")
        print(df.groupby('Cod.Curso')['Asignatura'].nunique().sort_index())

    if 'Cod.Car.Sec' in df.columns:
        df_local = df.copy()
        df_local['Carrera'] = df_local['Cod.Car.Sec'].astype(str).str.split('-').str[0]
        print("\n🎓 Distribución de estudiantes por carrera:")
        display(df_local['Carrera'].value_counts())

    if 'Aprobado' in df.columns:
        aprobacion_counts = df['Aprobado'].value_counts()
        print("\n✅ Estado de aprobación:")
        display(aprobacion_counts)
        print(f"\n📊 Tasa de aprobación: {aprobacion_counts.get('S', 0)/len(df)*100:.2f}%")

    return {
        'missing_data': missing_df,
        'year_semester_counts': year_semester_counts,
    }

# =============================================================================
# 3. VISUALIZACIONES
# =============================================================================

def visualizar_datos(df):
    """Genera visualizaciones básicas del dataset final."""
    print("\n" + "="*80)
    print("📈 VISUALIZACIONES DE DATOS")
    print("="*80)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    if 'Anho' in df.columns:
        df['Anho'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='skyblue')
        axes[0,0].set_title('Distribución de Registros por Año')
        axes[0,0].set_xlabel('Año')
        axes[0,0].set_ylabel('Número de Registros')
        axes[0,0].tick_params(axis='x', rotation=45)

    if 'Carrera' in df.columns:
        df['Carrera'].value_counts().plot(kind='bar', ax=axes[0,1], color='lightgreen')
        axes[0,1].set_title('Distribución por Carrera')
        axes[0,1].set_xlabel('Carrera')
        axes[0,1].set_ylabel('Número de Registros')
        axes[0,1].tick_params(axis='x', rotation=45)

    if 'Aprobado' in df.columns:
        df['Aprobado'].value_counts().plot(kind='pie', ax=axes[0,2], autopct='%1.1f%%', colors=['lightcoral', 'lightblue'])
        axes[0,2].set_title('Estado de Aprobación')
        axes[0,2].set_ylabel('')

    if 'Asignatura' in df.columns:
        df['Asignatura'].value_counts().head(10).plot(kind='barh', ax=axes[1,2], color='purple')
        axes[1,2].set_title('Top 10 Asignaturas con más Registros')
        axes[1,2].set_xlabel('Número de Registros')

    plt.tight_layout()
    plt.show()

    print("\n📊 Mapa de calor de datos faltantes:")
    plt.figure(figsize=(12, 8))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Mapa de Calor de Datos Faltantes')
    plt.show()

# =============================================================================
# 4. EJECUCIÓN PRINCIPAL
# =============================================================================

def main():
    """Carga los archivos del directorio actual y devuelve (datos, resultados)."""
    base_path = Path("/Users/marcosm2137/Documents/ProjectsML/ML1-2026.2_MarcosMaldonado/1er_Parcial")
    print(f"📂 Directorio de trabajo: {base_path}")

    files = sorted(base_path.glob('*.xlsx')) + sorted(base_path.glob('*.xls'))
    files = sorted(set(files), key=lambda p: p.name)

    print(f"🔍 Se encontraron {len(files)} archivos Excel para procesar")

    if not files:
        print("❌ No se encontraron archivos XLS/XLSX en este directorio.")
        return pd.DataFrame(), {}

    print("\n" + "="*80)
    print("🔄 PROCESO DE CARGA Y CONCATENACIÓN")
    print("="*80)

    datos = cargar_y_concatenar(str(base_path), [f.name for f in files])

    if datos.empty:
        print("❌ No se pudieron cargar datos")
        return pd.DataFrame(), {}

    print("\n" + "="*80)
    print("📊 ANÁLISIS DESCRIPTIVO GENERAL")
    print("="*80)
    resultados = analisis_descriptivo_general(datos)

    output_path = base_path / 'datos_concatenados.csv'
    datos.to_csv(output_path, index=False)
    print(f"\n✅ Datos concatenados guardados en '{output_path.name}'")

    print("\n" + "="*80)
    print("📋 RESUMEN FINAL")
    print("="*80)
    print(f"""
    ✅ PROCESO COMPLETADO EXITOSAMENTE

    📊 Estadísticas del dataset:
    • Registros totales: {len(datos):,}
    • Estudiantes únicos: {datos['ALUMNO_ID'].nunique() if 'ALUMNO_ID' in datos.columns else 'N/A'}
    • Asignaturas únicas: {datos['Asignatura'].nunique() if 'Asignatura' in datos.columns else 'N/A'}
    • Años académicos: {datos['Anho'].min()} - {datos['Anho'].max()}
    • Semestres: {', '.join(map(str, sorted(datos['Semestre'].unique())))}
    • Carreras: {datos['Carrera'].nunique() if 'Carrera' in datos.columns else 'N/A'}

    📁 Archivos generados:
    • {output_path.name}
    """)

    return datos, resultados

# =============================================================================
# EJECUTAR EL ANÁLISIS
# =============================================================================

if __name__ == "__main__":
    datos_procesados, resultados_analisis = main()

📂 Directorio de trabajo: /Users/marcosm2137/Documents/ProjectsML/ML1-2026.2_MarcosMaldonado/1er_Parcial
🔍 Se encontraron 4 archivos Excel para procesar

🔄 PROCESO DE CARGA Y CONCATENACIÓN
📄 Procesando archivo 1/4: rendimiento_año_2024_ciclo_2_anon.xlsx
  ✅ Registros agregados: 24,468 | Total acumulado: 24,468
📄 Procesando archivo 2/4: rendimiento_año_2025_ciclo_1_anon.xlsx
  ✅ Registros agregados: 19,407 | Total acumulado: 43,875
📄 Procesando archivo 3/4: rendimiento_año_2025_ciclo_2_anon.xlsx
  ✅ Registros agregados: 20,420 | Total acumulado: 64,295
📄 Procesando archivo 4/4: rendimiento_año_2026_ciclo_1_anon.xlsx
  ✅ Registros agregados: 8,445 | Total acumulado: 72,740

✅ Total de registros concatenados: 72,740

📊 ANÁLISIS DESCRIPTIVO GENERAL
📊 ANÁLISIS DESCRIPTIVO GENERAL DE LOS DATOS

1. INFORMACIÓN BÁSICA DEL DATASET
📌 Total de registros: 72,740
📌 Total de columnas: 30
📌 Total de estudiantes únicos: 6596
📌 Total de asignaturas únicas: 344
📌 Rango de años: 2024 - 2026
📌 Semestres di

,ALUMNO_ID,Cod.Asign,Asignatura,Cod.Car.Sec,Cod.Curso,Convocatoria,Anho,Semestre,Doc.Firma,Aprobado,Anho.Firma,Primer.Par,Segundo.Par,Tercer.Par,TPLab.,Lab.,Proy.,Pond.PP,Pond.SP,Pond.TPLab,Pond.Lab,Pond.Proy,Asis,Requisito,Firma,Primer.Rec,Segundo.Rec,Nota.Final,Archivo_Origen,FirmaCalculada
0,FIUNA_ALUMNO_005925,13012,FISICA 3,CGF-PLS13,3,2,2024,2,0,N,0,33,43,0,40,100,0,40,40,10,10,0,1,1,0.0,28,0,NaN,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN
1,FIUNA_ALUMNO_007516,13012,FISICA 3,CGF-PLS13,3,2,2024,2,0,N,0,33,43,0,40,100,0,40,40,10,10,0,1,1,0.0,28,0,NaN,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN
2,FIUNA_ALUMNO_005493,13012,FISICA 3,ELE-PLS13,3,2,2024,2,0,N,0,5,10,0,20,80,0,40,40,10,10,0,1,0,0.0,0,0,NaN,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN
3,FIUNA_ALUMNO_007517,13012,FISICA 3,ELE-PLS13,3,2,2024,2,0,N,0,5,10,0,20,80,0,40,40,10,10,0,1,0,0.0,0,0,NaN,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN
4,FIUNA_ALUMNO_006375,13012,FISICA 3,CIV-PLS13,3,2,2024,2,0,N,0,30,30,0,50,100,0,40,40,10,10,0,1,1,0.0,14,0,NaN,rendimiento_año_2024_ciclo_2_anon.xlsx,NaN



📋 Últimas 5 filas:


,ALUMNO_ID,Cod.Asign,Asignatura,Cod.Car.Sec,Cod.Curso,Convocatoria,Anho,Semestre,Doc.Firma,Aprobado,Anho.Firma,Primer.Par,Segundo.Par,Tercer.Par,TPLab.,Lab.,Proy.,Pond.PP,Pond.SP,Pond.TPLab,Pond.Lab,Pond.Proy,Asis,Requisito,Firma,Primer.Rec,Segundo.Rec,Nota.Final,Archivo_Origen,FirmaCalculada
72735,FIUNA_ALUMNO_8791969,5748,INTRODUCCION A LA INGENIERIA SATELITAL,MCT9-OPT,13,1,2026,1,39249,S,2026,0,0,0,95,0,80,0,0,20,0,0,1,1,NaN,0,0,2F-5,rendimiento_año_2026_ciclo_1_anon.xlsx,NaN
72736,FIUNA_ALUMNO_8930775,5748,INTRODUCCION A LA INGENIERIA SATELITAL,MCT9-OPT,13,1,2026,1,39249,N,2026,0,0,0,78,0,80,0,0,20,0,0,1,1,NaN,0,0,NaN,rendimiento_año_2026_ciclo_1_anon.xlsx,NaN
72737,FIUNA_ALUMNO_9200039,5748,INTRODUCCION A LA INGENIERIA SATELITAL,MCT9-OPT,13,1,2026,1,39249,S,2026,0,0,0,95,0,80,0,0,20,0,0,1,1,NaN,0,0,2F-5,rendimiento_año_2026_ciclo_1_anon.xlsx,NaN
72738,FIUNA_ALUMNO_9463375,5748,INTRODUCCION A LA INGENIERIA SATELITAL,MCT9-OPT,13,1,2026,1,39249,S,2026,0,0,0,95,0,80,0,0,20,0,0,1,1,NaN,0,0,2F-5,rendimiento_año_2026_ciclo_1_anon.xlsx,NaN
72739,FIUNA_ALUMNO_10658716,5748,INTRODUCCION A LA INGENIERIA SATELITAL,MCT9-OPT,13,1,2026,1,39249,S,2026,0,0,0,90,0,80,0,0,20,0,0,1,1,NaN,0,0,2F-5,rendimiento_año_2026_ciclo_1_anon.xlsx,NaN



📋 Tipos de datos por columna:
ALUMNO_ID             str
Cod.Asign           int64
Asignatura            str
Cod.Car.Sec           str
Cod.Curso           int64
Convocatoria        int64
Anho                int64
Semestre            int64
Doc.Firma           int64
Aprobado              str
Anho.Firma          int64
Primer.Par          int64
Segundo.Par         int64
Tercer.Par          int64
TPLab.              int64
Lab.                int64
Proy.               int64
Pond.PP             int64
Pond.SP             int64
Pond.TPLab          int64
Pond.Lab            int64
Pond.Proy           int64
Asis                int64
Requisito           int64
Firma             float64
Primer.Rec          int64
Segundo.Rec         int64
Nota.Final            str
Archivo_Origen        str
FirmaCalculada    float64
dtype: object

3. ESTADÍSTICAS DESCRIPTIVAS

📊 Estadísticas de columnas numéricas:


...


📚 Número de asignaturas por nivel:
Cod.Curso
1      6
2      5
3     18
4     25
5     50
6     60
7     42
8     40
9     36
10    20
13    78
Name: Asignatura, dtype: int64

🎓 Distribución de estudiantes por carrera:


Carrera
CIV           30313
ELE           12716
MCT            7620
IND            6881
CGF            5200
MEC            2697
INT9CONSTR     2136
ECA            1940
INT9MECANI      568
INT9ELECTR      552
MCT9            421
INT9TRANSP      389
INT9G           262
MEC9            250
INT9ORTERR      224
INT9SDIGYT      176
INT9            144
INT9SANEHI       99
INT9RNYMA        93
ECA9             59
Name: count, dtype: int64


✅ Estado de aprobación:


Aprobado
S    44384
N    28356
Name: count, dtype: int64


📊 Tasa de aprobación: 61.02%

✅ Datos concatenados guardados en 'datos_concatenados.csv'

📋 RESUMEN FINAL

    ✅ PROCESO COMPLETADO EXITOSAMENTE

    📊 Estadísticas del dataset:
    • Registros totales: 72,740
    • Estudiantes únicos: 6596
    • Asignaturas únicas: 344
    • Años académicos: 2024 - 2026
    • Semestres: 1, 2
    • Carreras: N/A

    📁 Archivos generados:
    • datos_concatenados.csv
    
